# Hyperparameter tuning

Each experiment changes one setting from `BASE` and validates on fold 0 of `results/csvs/data/folds.csv` (made by `cross_validation.ipynb`). Set `RUN_TRAINING = True` to train.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

import json
import warnings
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

import src.fine_tune.train as train_module
from src.fine_tune import Trainer, HybridModel
from src.fine_tune.losses import YOLOLoss
from src.data.preprocessing import ModelData, train_transform, eval_transform

from src.utils import parse_boxes, collate_fn, print_evaluation
from src.config.paths import CSV_DIR, PLOT_DIR, MODEL_DIR, DATASET_DIR, RESULTS_DIR
from src.config.types import TrainingObjects

In [ ]:
SEED = 42

# settings of scripts/training/train.py and run 20261028_231315
BASE = {"learning_rate": 1e-3, "weight_decay": 1e-4, "backbone_lr_scale": 0.1, "augmentation": True}

# each changes one setting from BASE, {} is BASE as the reference
EXPERIMENTS = [
    {},
    {"learning_rate": 3e-4}, {"learning_rate": 1e-4},
    {"learning_rate": 2e-3}, {"learning_rate": 3e-3}, # 1e-3 was the highest rate tried on  Mac and still won
    {"weight_decay": 1e-5}, {"weight_decay": 1e-3},
    {"backbone_lr_scale": 1.0},
    {"augmentation": False},
]

BATCH_SIZE = 4 # 8 doesn't fit on the Mac at 768px
ACCUMULATION_STEPS = 8 # 32 images per step; same as train.py
EPOCHS = 8
TRAIN_IMAGES = 3000
VAL_IMAGES = 1000
VAL_FOLD = 0
NUM_WORKERS = 0 # 8 on a CUDA PC

RUN_TRAINING = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# train_model logs CUDA memory but Mac doesnt support; ignore errors
if device.type != "cuda":
    torch.cuda.reset_peak_memory_stats = lambda: None
    torch.cuda.max_memory_allocated = lambda: 0

warnings.filterwarnings("ignore", message="CUDA is not available")
warnings.filterwarnings("ignore", message="Encountered more than 100 detections")

# save_results writes to MODEL_DIR, one folder per experiment named by what it changes from BASE
RUN_DIR = MODEL_DIR / "hyperparameter_tuning"
train_module.MODEL_DIR = RUN_DIR

TUNING_DIR = CSV_DIR / "hyperparameter_tuning"
PLOT_DIR = PLOT_DIR / "hyperparameter_tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(device, RUN_DIR)

## Existing runs

In [ ]:
def load_runs(folder):
    runs = {}
    for run_dir in sorted(folder.iterdir()):
        if not (run_dir / "history.csv").exists():
            continue

        with open(run_dir / "metadata.json") as f:
            meta = json.load(f)

        # older runs prefix the metrics with val_
        history = pd.read_csv(run_dir / "history.csv")
        history = history.rename(columns=lambda c: c if c == "val_loss" else c.removeprefix("val_"))
        runs[run_dir.name] = (meta, history)
    return runs


def short_label(note):
    # name a run by what it changes from BASE, so the reference run reads "base"
    parts = note.split(", val fold")[0].split(", ")
    base = [
        f"learning rate {BASE['learning_rate']:g}", f"weight decay {BASE['weight_decay']:g}",
        f"backbone lr x{BASE['backbone_lr_scale']:g}", f"augmentation {'on' if BASE['augmentation'] else 'off'}",
    ]
    if len(parts) != len(base):
        return note # older runs have free text notes
    return ", ".join(p for p, b in zip(parts, base) if p != b) or "base"


def plot_runs(runs, path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for name, (meta, history) in runs.items():
        batch_size = meta["parameters"]["batch_size"]
        label = short_label(meta["notes"][0])
        line, = axes[0].plot(history["epoch"], history["train_loss"] / batch_size)
        axes[0].plot(history["epoch"], history["val_loss"] / batch_size, "--", color=line.get_color())
        axes[1].plot(history["epoch"], history["map50_95"], marker="o", color=line.get_color(), label=label)

    axes[0].set_title("Loss per image (solid train, dashed val)")
    axes[1].set_title("Validation mAP50-95")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(linestyle="--", alpha=0.5)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(path)
    plt.show()


def summarise(runs):
    rows = []
    for name, (meta, history) in runs.items():
        best = history.loc[history["map50_95"].idxmax()]
        rows.append({
            "run": name,
            "experiment": meta["notes"][0],
            "notes": ", ".join(meta["notes"]),
            "learning_rate": meta["parameters"]["learning_rate"],
            "weight_decay": meta["parameters"]["weight_decay"],
            "epochs": len(history),
            "best_epoch": int(best["epoch"]),
            "best_map50": best["map50"],
            "best_map50_95": best["map50_95"],
            "best_recall50": best.get("recall50"),
            "best_mar_100": best.get("mar_100"),
        })
    return pd.DataFrame(rows)


historical = load_runs(MODEL_DIR)
plot_runs(historical, PLOT_DIR / "full_length_runs.svg")

historical_table = summarise(historical)
historical_table.to_csv(TUNING_DIR / "full_length_runs.csv", index=False)
historical_table.round(4)

## Data

In [ ]:
df = pd.read_csv(CSV_DIR / "data/metadata.csv")
df["boxes"] = df["boxes"].apply(lambda value: parse_boxes(json.loads(value)))

folds = pd.read_csv(CSV_DIR / "data/folds.csv")
df = df.merge(folds[["path", "fold"]], on="path")

train_df = df[df["fold"] != VAL_FOLD]
val_df = df[df["fold"] == VAL_FOLD]
print(f"{len(train_df)} train and {len(val_df)} val images to sample from")

In [ ]:
train_data = list(train_df.sample(TRAIN_IMAGES, random_state=SEED)[["path", "boxes"]].itertuples(index=False, name=None))
val_data = list(val_df.sample(VAL_IMAGES, random_state=SEED)[["path", "boxes"]].itertuples(index=False, name=None))

train_dataset = ModelData(
    DATASET_DIR, train_data, train_transform,
    flip_prob=0.5,
    zoom_prob=0.5, zoom_max=1.3
)
val_dataset = ModelData(DATASET_DIR, val_data, eval_transform)

# seeded in train_experiment so every run sees the same image order
generator = torch.Generator()
loader_args = dict(batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)

## Training

In [ ]:
torch.manual_seed(SEED)
initial_state = HybridModel().state_dict()

git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()

def describe(settings):
    return (
        f"learning rate {settings['learning_rate']:g}, weight decay {settings['weight_decay']:g}, "
        f"backbone lr x{settings['backbone_lr_scale']:g}, augmentation {'on' if settings['augmentation'] else 'off'}"
    )


def folder_name(settings):
    # readable run folder, named by what the experiment changes from BASE
    fmt = lambda value: f"{value:.0e}".replace("e-0", "e-")
    changed = []
    if settings["learning_rate"] != BASE["learning_rate"]:
        changed.append(f"lr_{fmt(settings['learning_rate'])}")
    if settings["weight_decay"] != BASE["weight_decay"]:
        changed.append(f"weight_decay_{fmt(settings['weight_decay'])}")
    if settings["backbone_lr_scale"] != BASE["backbone_lr_scale"]:
        changed.append(f"backbone_lr_x{settings['backbone_lr_scale']:g}")
    if settings["augmentation"] != BASE["augmentation"]:
        changed.append(f"augmentation_{'on' if settings['augmentation'] else 'off'}")
    return "_".join(changed) or "base"


def train_experiment(settings):
    model = HybridModel().to(device)
    model.load_state_dict(initial_state)

    optimiser = AdamW(
        [
            # pretrained backbone gets smaller steps so the ImageNet features arent wiped out
            {"params": model.backbone.parameters(), "lr": settings["learning_rate"] * settings["backbone_lr_scale"]},
            {"params": [p for name, p in model.named_parameters() if not name.startswith("backbone.")]},
        ],
        lr=settings["learning_rate"],
        weight_decay=settings["weight_decay"]
    )

    training_objs = TrainingObjects(
        model=model,
        optimizer=optimiser,
        criterion=YOLOLoss(model),
        scaler=torch.amp.GradScaler("cuda", enabled=device.type == "cuda"),
        device=device,
        accumulation_steps=ACCUMULATION_STEPS,
    )

    # workers restart each epoch, so they pick these up
    train_dataset.flip_prob = 0.5 if settings["augmentation"] else 0.0
    train_dataset.zoom_prob = 0.5 if settings["augmentation"] else 0.0

    torch.manual_seed(SEED)
    generator.manual_seed(SEED)

    return Trainer(objs=training_objs).train_model(epochs=EPOCHS, train_loader=train_loader, val_loader=val_loader)

In [ ]:
if RUN_TRAINING:
    done = [meta["notes"][0] for meta, _ in load_runs(RUN_DIR).values() if "completed" in meta["notes"]] if RUN_DIR.exists() else []

    for changes in EXPERIMENTS:
        settings = BASE | changes
        experiment = f"{describe(settings)}, val fold {VAL_FOLD}, train images {TRAIN_IMAGES}, val images {VAL_IMAGES}"
        if experiment in done:
            print(f"{describe(settings)} already saved, skipping")
            continue

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        print(describe(settings))
        results = train_experiment(settings)

        if results.error is None:
            best = results.history.eval[results.best_epoch - 1]
            print_evaluation(best)
            notes = [experiment, "completed", f"[smoke, fire] mAP50-95 {np.round(best.evaluation.map_per_class, 4).tolist()}"]
        else:
            notes = [experiment, f"failed: {results.error!r}"]

        model_info = {
            "timestamp": timestamp,
            "notes": notes,
            "git_commit": git_commit,
            "parameters": {
                "seed": SEED, "num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda",
                "batch_size": BATCH_SIZE, "epochs": EPOCHS, "accumulation_steps": ACCUMULATION_STEPS,
                "learning_rate": settings["learning_rate"], "weight_decay": settings["weight_decay"],
                "model_name": "not saved",
            },
        }

        results.best_state = None # no model.pt
        Trainer.save_results(results, model_info, folder_name(settings))

## Results

In [ ]:
if RUN_DIR.exists():
    candidates = load_runs(RUN_DIR)
    plot_runs(candidates, PLOT_DIR / "experiments.svg")

    table = summarise(candidates).sort_values("best_map50_95", ascending=False)
    table.to_csv(TUNING_DIR / "experiments.csv", index=False)
    display(table.round(4))
else:
    print("No runs yet, set RUN_TRAINING = True to train")

In [ ]:
if RUN_DIR.exists():
    completed = table[table["notes"].str.contains("completed")]
    best = completed["best_map50_95"].iloc[0] if len(completed) else 0.0
    runner_up = completed["best_map50_95"].iloc[1] if len(completed) > 1 else 0.0

    # one seed and a sample of val images, gaps under 0.01 count as noise
    if best < 0.01:
        print("No winner: every experiment is still near zero mAP, train for longer")
    elif best - runner_up < 0.01:
        print(f"Tie: {completed['experiment'].iloc[0]} is ahead by only {best - runner_up:.4f}")
    else:
        print(f"Best: {completed['experiment'].iloc[0]} (mAP50-95 {best:.4f}, ahead by {best - runner_up:.4f})")